In [ ]:

import torch
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def backtranslation_dataset(
    fichier_txt,
    nom_modele="facebook/nllb-200-distilled-600M",
    langue_source="fra_Latn",
    langue_cible="arb_Latn",
    test_size=0.10,
    val_size=0.10,
    batch_size=8,
    seed=42,
):
    # 1. Charger le corpus français
    with open(fichier_txt, "r", encoding="utf-8-sig") as f:
        phrases_fr = list(dict.fromkeys(
            ligne.strip() for ligne in f if ligne.strip()
        ))

    if len(phrases_fr) < 3:
        raise ValueError("Le fichier doit contenir au moins 3 phrases.")

    if test_size <= 0 or val_size <= 0 or test_size + val_size >= 1:
        raise ValueError("Les proportions doivent être positives et leur somme < 1.")

    # 2. Charger le modèle NLLB
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(
        nom_modele,
        src_lang=langue_source,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(nom_modele).to(device)
    model.eval()

    target_id = tokenizer.convert_tokens_to_ids(langue_cible)

    if target_id == tokenizer.unk_token_id:
        raise ValueError(f"Code de langue cible inconnu : {langue_cible}")

    # 3. Générer les traductions vers la langue cible
    phrases_arb = []

    with torch.inference_mode():
        for i in range(0, len(phrases_fr), batch_size):
            batch_fr = phrases_fr[i:i + batch_size]

            inputs = tokenizer(
                batch_fr,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            ).to(device)

            outputs = model.generate(
                **inputs,
                forced_bos_token_id=target_id,
                max_new_tokens=256,
                num_beams=4,
            )

            phrases_arb.extend(
                tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True,
                )
            )

            print(f"{min(i + batch_size, len(phrases_fr))}/{len(phrases_fr)}")

    # 4. Construire exactement la structure translation
    dataset_dict = {
        "translation": [
            {"arb": arb, "fr": fr}
            for arb, fr in zip(phrases_arb, phrases_fr)
        ]
    }

    dataset = Dataset.from_dict(dataset_dict)

    # 5. Créer train, val et test
    split_test = dataset.train_test_split(
        test_size=test_size,
        seed=seed,
    )

    train_val = split_test["train"].train_test_split(
        test_size=val_size / (1 - test_size),
        seed=seed,
    )

    resultat = DatasetDict({
        "train": train_val["train"],
        "val": train_val["test"],
        "test": split_test["test"],
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(resultat)
    print("\nExemple :", resultat["train"][0])

    return resultat


### Utilisation

In [ ]:
dataset_dict_not_bible = backtranslation_dataset(
    fichier_txt=PATH + "french_not_bible.txt",
    nom_modele="facebook/nllb-200-distilled-600M",
    langue_source="fra_Latn",
    langue_cible="arb_Arab",
)

### Concater avec le datadict existant

In [ ]:
from datasets import concatenate_datasets, DatasetDict

dataset_combine = DatasetDict({
    split: concatenate_datasets([
        train_test_val_dataset_bible[split],
        dataset_dict_not_bible[split],
    ]).shuffle(seed=42)
    for split in ["train", "val", "test"]
})

In [ ]:
dataset_combine.save_to_disk(Pathe)